In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install -U trl
!pip install -q trl transformers accelerate datasets

In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_excel("/content/cleaned_dataset.xlsx")
val_df = pd.read_csv("validation.csv")

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)

In [ ]:
def format_example(example):

    messages = [
        {
            "role": "system",
            # "content": "You are answering questions about Vishnu."
           "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
        },
        {
            "role": "user",
            "content": example["Clean Question"]
        },
        {
            "role": "assistant",
            "content": example["Clean Answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [ ]:
import torch
import transformers
import peft
import bitsandbytes as bnb

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cpu
CUDA: None
Transformers: 5.12.0
PEFT: 0.19.1
BitsAndBytes: 0.49.2


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./full_ft_qwen",

    num_train_epochs=20,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_steps=1,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
train_dataset = train_dataset.map(tokenize_function)
val_dataset = val_dataset.map(tokenize_function)

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.652753,0.337071
2,0.204857,0.264273
3,0.211798,0.212661
4,0.181705,0.166737


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=300, training_loss=0.14939187589411934, metrics={'train_runtime': 1400.4924, 'train_samples_per_second': 1.642, 'train_steps_per_second': 0.214, 'total_flos': 1264382450073600.0, 'train_loss': 0.14939187589411934, 'epoch': 20.0})

In [ ]:
trainer.save_model("./full_ft_qwen")
tokenizer.save_pretrained("./full_ft_qwen")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./full_ft_qwen/tokenizer_config.json',
 './full_ft_qwen/chat_template.jinja',
 './full_ft_qwen/tokenizer.json')

In [ ]:
!nvidia-smi

Thu Jun 25 07:42:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             30W /   70W |    6103MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
question="What's your father's name?"

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
    },
    {
        "role": "user",
        "content": question
    }
]

NameError: name 'question' is not defined

In [ ]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
 You are Vishnu's personal AI assistant.

Always answer using the information you have learned about Vishnu.

Do not invent facts.

If you do not know the answer, say you don't know.

Answer as Vishnu in first person.
user
What's your father's name?
assistant
My father's name is Amarapu Ramaiah.


In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(folder_path="./full_ft_qwen", repo_id="vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-sft", repo_type="model")


Mounted at /content/drive


In [ ]:
!cp -r /content/my_model /content/drive/MyDrive/my_model

# Finetunig with SFT trainer


In [ ]:
!pip install -q trl transformers accelerate datasets

In [ ]:
!pip install -U trl

In [ ]:
def format_example(example):

    messages = [
        {
            "role": "system",
            "content": (
                "You are Vishnu's personal AI assistant. "
                "Answer questions about Vishnu using the provided information."
            )
        },
         {
            "role": "user",
            "content": example["Clean Question"]
        },
        {
            "role": "assistant",
            "content": example["Clean Answer"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

In [ ]:
train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./full_ft_qwen",

    num_train_epochs=20,

    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=1,

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
from trl import SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)


Adding EOS to train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/115 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.007489,1.059699,1.191406,0.776741,7782.000000
2,0.866934,0.814485,0.928906,0.821536,15564.000000
3,0.773047,0.624819,0.805859,0.854236,23346.000000
4,0.676454,0.491114,0.671582,0.888716,31128.000000
5,0.512046,0.312358,0.506641,0.930923,38910.000000
6,0.193911,0.247497,0.416797,0.949270,46692.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=300, training_loss=0.33655707637468973, metrics={'train_runtime': 1027.4067, 'train_samples_per_second': 2.239, 'train_steps_per_second': 0.292, 'total_flos': 334219572910080.0, 'train_loss': 0.33655707637468973, 'epoch': 20.0})

In [ ]:
trainer.save_model("./full_ft_qwen_SFT")
tokenizer.save_pretrained("./full_ft_qwen_SFT")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./full_ft_qwen_SFT/tokenizer_config.json',
 './full_ft_qwen_SFT/chat_template.jinja',
 './full_ft_qwen_SFT/tokenizer.json')

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "./full_ft_qwen_SFT",
    device_map="auto"
)
messages = [
    {
        "role": "system",
        "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
    },
    {
        "role": "user",
        "content": "What is my father name?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

system
You are Vishnu's personal AI assistant. Answer questions about Vishnu.
user
What is my father name?
assistant
Your father's name is Amarapu Dhanamma.


In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(folder_path="./full_ft_qwen_SFT", repo_id="vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-sft", repo_type="model")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t_qwen_SFT/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...wen_SFT/model.safetensors:   0%|          |  610kB /  988MB            

  ...wen_SFT/training_args.bin:   9%|8         |   484B / 5.65kB            

CommitInfo(commit_url='https://huggingface.co/vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-sft/commit/018a0cb25d9b7e4de72734f3fd3466755208ddc7', commit_message='Upload folder using huggingface_hub', commit_description='', oid='018a0cb25d9b7e4de72734f3fd3466755208ddc7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-sft', endpoint='https://huggingface.co', repo_type='model', repo_id='vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-sft'), pr_revision=None, pr_num=None)

# Starting With Lora


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    bias="none",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, peft_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./full_ft_qwen",

    num_train_epochs=20,

    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=1,

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.816216,1.445794,1.075889,29440.000000,0.764706
2,0.673607,0.800193,0.739185,58880.000000,0.836471
3,0.488555,0.487306,0.497340,88320.000000,0.910588
4,0.393181,0.388220,0.396386,117760.000000,0.924314
5,0.422065,0.365391,0.373740,147200.000000,0.929020
6,0.252250,0.354062,0.359822,176640.000000,0.932157
7,0.170421,0.345679,0.347127,206080.000000,0.931373
8,0.269041,0.338563,0.347406,235520.000000,0.932549
9,0.231072,0.332232,0.339858,264960.000000,0.933725
10,0.383187,0.326212,0.332941,294400.000000,0.933725


TrainOutput(global_step=300, training_loss=0.5624946024020513, metrics={'train_runtime': 915.1683, 'train_samples_per_second': 2.513, 'train_steps_per_second': 0.328, 'total_flos': 1295464759296000.0, 'train_loss': 0.5624946024020513, 'epoch': 20.0})

In [ ]:
trainer.model.save_pretrained("qwen-personal-lora")
tokenizer.save_pretrained("qwen-personal-lora")

('qwen-personal-lora/tokenizer_config.json',
 'qwen-personal-lora/chat_template.jinja',
 'qwen-personal-lora/tokenizer.json')

NameError: name 'model_name' is not defined

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
    },
    {
        "role": "user",
        "content": "What's your name?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Vishnu's personal AI assistant. Answer questions about Vishnu.
user
What's your name?
assistant
I am Vishnu, a 21-year-old software engineer from Bangalore, India.


In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(folder_path="/content/qwen-personal-lora", repo_id="vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-LORA", repo_type="model")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 38.0kB / 35.2MB            

  ...sonal-lora/tokenizer.json:  99%|#########8| 11.3MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-LORA/commit/095647442739710967cab6569976e1d00d10daab', commit_message='Upload folder using huggingface_hub', commit_description='', oid='095647442739710967cab6569976e1d00d10daab', pr_url=None, repo_url=RepoUrl('https://huggingface.co/vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-LORA', endpoint='https://huggingface.co', repo_type='model', repo_id='vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-LORA'), pr_revision=None, pr_num=None)

#QLORA


In [ ]:
!pip uninstall -y bitsandbytes

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2


In [ ]:
!pip install -U bitsandbytes

  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


In [ ]:
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from transformers import BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
import torch
import transformers
import peft
import bitsandbytes as bnb

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA: 12.8
Transformers: 5.12.0
PEFT: 0.19.1
BitsAndBytes: 0.49.2
GPU: Tesla T4


In [ ]:
from peft import get_peft_model

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./full_ft_qwen",

    num_train_epochs=20,

    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=1,

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.211228,0.911360,0.673824,29440.000000,0.860392
2,0.472291,0.615164,0.595225,58880.000000,0.896471
3,0.419361,0.423709,0.467132,88320.000000,0.923137
4,0.374739,0.383628,0.389252,117760.000000,0.927451


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.211228,0.911360,0.673824,29440.000000,0.860392
2,0.472291,0.615164,0.595225,58880.000000,0.896471
3,0.419361,0.423709,0.467132,88320.000000,0.923137
4,0.374739,0.383628,0.389252,117760.000000,0.927451
5,0.434324,0.370461,0.371232,147200.000000,0.930588
6,0.259366,0.360558,0.357600,176640.000000,0.929804
7,0.172788,0.354882,0.352653,206080.000000,0.930588
8,0.268328,0.348111,0.349091,235520.000000,0.931373
9,0.248493,0.342927,0.344588,264960.000000,0.931765
10,0.395776,0.337392,0.337285,294400.000000,0.933333


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=300, training_loss=0.45103487009803456, metrics={'train_runtime': 1441.1824, 'train_samples_per_second': 1.596, 'train_steps_per_second': 0.208, 'total_flos': 1295464759296000.0, 'train_loss': 0.45103487009803456, 'epoch': 20.0})

In [ ]:
trainer.model.save_pretrained("qwen-personal-qlora")
tokenizer.save_pretrained("qwen-personal-qlora")

('qwen-personal-qlora/tokenizer_config.json',
 'qwen-personal-qlora/chat_template.jinja',
 'qwen-personal-qlora/tokenizer.json')

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
    },
    {
        "role": "user",
        "content": "Who are you?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Vishnu's personal AI assistant. Answer questions about Vishnu.
user
Who are you?
assistant
I am a 13th year student at IIT Kgp, and I'm here to learn from you.


In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
    },
    {
        "role": "user",
        "content": "What's your father's name?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Vishnu's personal AI assistant. Answer questions about Vishnu.
user
What's your father's name?
assistant
My father's name is Ramesh Kumar Singh.


In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(folder_path="/content/qwen-personal-qlora", repo_id="vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-QLora", repo_type="model")


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6a3e1be2-7f6b60875513b27c234e5b25;8fc03889-7e6a-44a1-b52d-100bc2740749)

Repository Not Found for url: https://huggingface.co/api/models/vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-QLora/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.